# Практическая работа №3: Обработка выборочных данных. Нахождение интервальных оценок параметров распределения. Проверка статистической гипотезы о нормальном распределении

Выполнили студенты гр. 2384 Исмаилов Максим Владимирович и Дамакин Роман Павлович. Вариант №31.

## Цель работы

Получение практических навыков вычисления интервальных статистических оценок параметров распределения выборочных данных и проверки «справедливости» статистических гипотез.

## Постановка задачи

Для заданной надежности определить (на основании выборочных данных и результатов выполнения практической работы №2) границы доверительных интервалов для математического ожидания и среднеквадратичного отклонения случайной величины. Проверить гипотезу о нормальном распределении исследуемой случайной величины с помощью критерия Пирсона $\chi^2$. Дать содержательную интерпретацию полученным результатам.

## Порядок выполнения работы

1. Вычислить точность и доверительный интервал для математического ожидания при неизвестном среднеквадратичном отклонении при заданном объёме выборки для доверительной точности $\gamma \in \{0.95, 0.99\}$. Сделать выводы.
2. Для вычисления границ доверительного интервала для среднеквадратичного отклонения определить значение $q$ при заданных $\gamma$ и $n$. Построить доверительные интервалы, сделать выводы.
3. Проверить гипотезу о нормальности заданного распределения с помощью критерия $\chi^2$ (Пирсона). Для этого необходимо найти теоретические частоты и вычислить наблюдаемое значение критерия. Для удобства вычисления необходимо заполнить табл. 1.
4. Доказать, что
$$\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n.$$
Проконтролировать корректность вычисления $\chi^2_{\text{набл}}$.

5. По заданному уровню значимости $\alpha = 0.05$ и числу степеней свободы $df$ найти критическую точку $\chi^2_{\text{крит}}$ и сравнить с наблюдаемым значением. Сделать выводы.

## Основные теоретические положения

**Критерий согласия Пирсона $\chi^2$** используется для проверки гипотезы о нормальном распределении генеральной совокупности:

$$
\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{(n_i - n_i')^2}{n_i'},
\qquad
n_i' = n \cdot p_i
$$

где $n_i$ — наблюдаемая частота, а $n_i'$ — теоретическая частота для $i$-го интервала.  
Число степеней свободы определяется по формуле:

$$
df = k - 3
$$

Гипотеза $H_0$ о нормальности распределения принимается, если выполняется условие:

$$
\chi^2_{\text{набл}} < \chi^2_{\text{крит}}
$$

**Доверительный интервал** — это диапазон значений, который с заданной доверительной вероятностью $\gamma$ содержит истинное значение оцениваемого параметра.

**Доверительный интервал для математического ожидания** при неизвестном значении $\sigma$ строится на основе распределения Стьюдента:

$$
\bar{x}_в - \varepsilon < \mu < \bar{x}_в + \varepsilon,
\qquad
\varepsilon = t_{\gamma,\, n-1} \cdot \frac{s}{\sqrt{n}}
$$

**Доверительный интервал для среднеквадратичного отклонения** определяется с использованием $\chi^2$-распределения:

$$
s \cdot q_1 < \sigma < s \cdot q_2
$$

где

$$
q_1 = \sqrt{\frac{n-1}{\chi^2_{1-\alpha/2}}},
\qquad
q_2 = \sqrt{\frac{n-1}{\chi^2_{\alpha/2}}}
$$

## Выполнение работы


In [2]:

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown


def md(text: str) -> None:
    display(Markdown(text))


def md_table(df: pd.DataFrame, digits: int = 4) -> None:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.{digits}f}")
    display(Markdown(out.to_markdown(index=False)))


csv_path = 'sample.csv'
raw = pd.read_csv(csv_path, skiprows=3)
raw.columns = ['nu', 'E']
raw = raw.astype(float)

n = len(raw)
alpha = 0.05

md(f'Объём выборки: **n = {n}**')
display(raw.head())


Объём выборки: **n = 115**

,nu,E
0,480.0,153.3
1,510.0,129.4
2,426.0,119.0
3,482.0,139.9
4,393.0,103.2


In [3]:
summary_rows = []
for name in ['nu', 'E']:
    x = raw[name].to_numpy()
    summary_rows.append({
        'Признак': name,
        'n': len(x),
        'x̄': x.mean(),
        's²': x.var(ddof=1),
        's': x.std(ddof=1),
        'min': x.min(),
        'max': x.max()
    })

summary_df = pd.DataFrame(summary_rows)
md_table(summary_df)

| Признак   |   n |       x̄ |      s² |       s |   min |   max |
|:----------|----:|--------:|--------:|--------:|------:|------:|
| nu        | 115 | 448.722 | 3446.15 | 58.7039 | 320   | 593   |
| E         | 115 | 127.13  |  588.99 | 24.2691 |  64.5 | 187.4 |

## 1. Доверительный интервал для математического ожидания

Если значение стандартного отклонения $\sigma$ заранее неизвестно, то величина погрешности оценки определяется по формуле:

$$
\varepsilon = t_{\gamma,\, n-1} \cdot \frac{s}{\sqrt{n}}
$$

Тогда доверительный интервал для математического ожидания записывается в виде:

$$
\left(\bar{x}_в - \varepsilon,\; \bar{x}_в + \varepsilon\right)
$$

Построение интервала выполняется для уровней доверия $\gamma \in \{0.95, 0.99\}$.

In [4]:
def mean_confidence_intervals(x: np.ndarray, gammas=(0.95, 0.99)) -> pd.DataFrame:
    x = np.asarray(x, dtype=float)
    n = len(x)
    xbar = x.mean()
    s = x.std(ddof=1)

    rows = []
    for gamma in gammas:
        t_value = stats.t.ppf((1 + gamma) / 2, df=n - 1)
        eps = t_value * s / np.sqrt(n)
        rows.append({
            'γ': gamma,
            't': t_value,
            'ε': eps,
            'Нижняя граница': xbar - eps,
            'Верхняя граница': xbar + eps,
        })
    return pd.DataFrame(rows)


nu_mean_ci = mean_confidence_intervals(raw['nu'].to_numpy())
e_mean_ci = mean_confidence_intervals(raw['E'].to_numpy())

print('Доверительный интервал для математического ожидания признака nu')
md_table(nu_mean_ci)
print('Доверительный интервал для математического ожидания признака E')
md_table(e_mean_ci)

Доверительный интервал для математического ожидания признака nu


|    γ |      t |       ε |   Нижняя граница |   Верхняя граница |
|-----:|-------:|--------:|-----------------:|------------------:|
| 0.95 | 1.981  | 10.8443 |          437.877 |           459.566 |
| 0.99 | 2.6196 | 14.3404 |          434.381 |           463.062 |

Доверительный интервал для математического ожидания признака E


|    γ |      t |      ε |   Нижняя граница |   Верхняя граница |
|-----:|-------:|-------:|-----------------:|------------------:|
| 0.95 | 1.981  | 4.4832 |          122.646 |           131.613 |
| 0.99 | 2.6196 | 5.9285 |          121.201 |           133.058 |

Для обоих признаков при переходе от $\gamma=0.95$ к $\gamma=0.99$ интервалы расширяются, так как возрастает требуемая надёжность оценки. Это полностью соответствует теории: чем выше доверительная вероятность, тем больше точность по надёжности и тем шире доверительный интервал.

## 2. Доверительный интервал для среднеквадратичного отклонения

Для построения доверительного интервала для параметра $\sigma$ используются коэффициенты $q_1$ и $q_2$, которые находятся по $\chi^2$-распределению при числе степеней свободы $n-1$:

$$
q_1 = \sqrt{\frac{n-1}{\chi^2_{1-\alpha/2,\, n-1}}}, \qquad
q_2 = \sqrt{\frac{n-1}{\chi^2_{\alpha/2,\, n-1}}}
$$

Тогда границы доверительного интервала для среднеквадратичного отклонения задаются неравенством:

$$
s \cdot q_1 < \sigma < s \cdot q_2
$$

In [5]:
def sigma_confidence_intervals(x: np.ndarray, gammas=(0.95, 0.99)) -> pd.DataFrame:
    x = np.asarray(x, dtype=float)
    n = len(x)
    s2 = x.var(ddof=1)

    rows = []
    for gamma in gammas:
        q_left = stats.chi2.ppf((1 - gamma) / 2, df=n - 1)
        q_right = stats.chi2.ppf((1 + gamma) / 2, df=n - 1)
        sigma_low = np.sqrt((n - 1) * s2 / q_right)
        sigma_high = np.sqrt((n - 1) * s2 / q_left)
        rows.append({
            'γ': gamma,
            'q1 = χ²((1-γ)/2; n-1)': q_left,
            'q2 = χ²((1+γ)/2; n-1)': q_right,
            'Нижняя граница σ': sigma_low,
            'Верхняя граница σ': sigma_high,
        })
    return pd.DataFrame(rows)


nu_sigma_ci = sigma_confidence_intervals(raw['nu'].to_numpy())
e_sigma_ci = sigma_confidence_intervals(raw['E'].to_numpy())

print('Доверительный интервал для СКО признака nu')
md_table(nu_sigma_ci)
print('Доверительный интервал для СКО признака E')
md_table(e_sigma_ci)

Доверительный интервал для СКО признака nu


|    γ |   q1 = χ²((1-γ)/2; n-1) |   q2 = χ²((1+γ)/2; n-1) |   Нижняя граница σ |   Верхняя граница σ |
|-----:|------------------------:|------------------------:|-------------------:|--------------------:|
| 0.95 |                 86.3425 |                 145.441 |            51.9727 |             67.4539 |
| 0.99 |                 78.8618 |                 156.637 |            50.0809 |             70.5808 |

Доверительный интервал для СКО признака E


|    γ |   q1 = χ²((1-γ)/2; n-1) |   q2 = χ²((1+γ)/2; n-1) |   Нижняя граница σ |   Верхняя граница σ |
|-----:|------------------------:|------------------------:|-------------------:|--------------------:|
| 0.95 |                 86.3425 |                 145.441 |            21.4863 |             27.8865 |
| 0.99 |                 78.8618 |                 156.637 |            20.7042 |             29.1792 |

Интервалы для $\sigma$ также расширяются при увеличении доверительной вероятности. Для признака **nu** интервал шире, чем для **E**, что связано с большим абсолютным разбросом значений объёмного веса по сравнению с модулем упругости.

## 3. Проверка гипотезы о нормальности (критерий Пирсона $\chi^2$)

В качестве нулевой гипотезы принимается предположение, что исследуемая случайная величина подчиняется нормальному распределению:

$$
H_0: \text{случайная величина имеет нормальное распределение}
$$

Ожидаемые (теоретические) частоты для каждого интервала вычисляются по формуле:

$$
n_i' = n \cdot \left[\Phi(x_{i+1}) - \Phi(x_i)\right]
$$

где $\Phi$ — функция нормального распределения, заданная параметрами выборочного среднего $\bar{x}_в$ и выборочного среднеквадратичного отклонения $s$.

Если для некоторых интервалов теоретические частоты оказываются меньше 5, такие интервалы объединяют с соседними.

In [7]:
def make_interval_edges(x: np.ndarray) -> np.ndarray:
    n = len(x)
    k = int(round(1 + 3.322 * np.log10(n)))
    return np.linspace(x.min(), x.max(), k + 1)


def merge_small_expected(counts: np.ndarray, expected: np.ndarray, labels: list[str], min_expected: float = 5.0):
    groups = []
    start = 0
    obs_acc = float(counts[0])
    exp_acc = float(expected[0])

    for i in range(1, len(counts)):
        if exp_acc < min_expected:
            obs_acc += float(counts[i])
            exp_acc += float(expected[i])
        else:
            groups.append((start, i - 1, obs_acc, exp_acc))
            start = i
            obs_acc = float(counts[i])
            exp_acc = float(expected[i])
    groups.append((start, len(counts) - 1, obs_acc, exp_acc))

    if len(groups) >= 2 and groups[-1][3] < min_expected:
        a0, b0, n0, e0 = groups[-2]
        a1, b1, n1, e1 = groups[-1]
        groups[-2] = (a0, b1, n0 + n1, e0 + e1)
        groups.pop()

    merged_labels, merged_obs, merged_exp = [], [], []
    for a, b, n_obs, n_exp in groups:
        if a == b:
            merged_labels.append(labels[a])
        else:
            merged_labels.append(f'{labels[a]} ∪ {labels[b]}')
        merged_obs.append(n_obs)
        merged_exp.append(n_exp)

    return merged_labels, np.array(merged_obs), np.array(merged_exp)


def pearson_table(x: np.ndarray, name: str) -> tuple[pd.DataFrame, dict]:
    x = np.asarray(x, dtype=float)
    n = len(x)
    mu = x.mean()
    sigma = x.std(ddof=1)
    edges = make_interval_edges(x)
    counts, _ = np.histogram(x, bins=edges)

    labels = []
    probs = []
    for i in range(len(edges) - 1):
        a, b = edges[i], edges[i + 1]
        if i == 0:
            p = stats.norm.cdf((b - mu) / sigma)
            labels.append(f'(-∞, {b:.2f}]')
        elif i == len(edges) - 2:
            p = 1 - stats.norm.cdf((a - mu) / sigma)
            labels.append(f'[{a:.2f}, +∞)')
        else:
            p = stats.norm.cdf((b - mu) / sigma) - stats.norm.cdf((a - mu) / sigma)
            labels.append(f'[{a:.2f}, {b:.2f})')
        probs.append(p)

    expected = n * np.array(probs)
    merged_labels, merged_obs, merged_exp = merge_small_expected(counts, expected, labels)

    table = pd.DataFrame({
        'i': np.arange(1, len(merged_labels) + 1),
        '[x_i, x_{i+1})': merged_labels,
        'n_i': merged_obs,
        'p_i': merged_exp / n,
        "n'_i": merged_exp,
    })
    table["(n_i - n'_i)^2"] = (table["n_i"] - table["n'_i"]) ** 2
    table["(n_i - n'_i)^2 / n'_i"] = table["(n_i - n'_i)^2"] / table["n'_i"]
    table["n_i^2"] = table["n_i"] ** 2
    table["n_i^2 / n'_i"] = table["n_i^2"] / table["n'_i"]

    sigma_row = pd.DataFrame([{
        'i': 'Σ',
        '[x_i, x_{i+1})': '-',
        'n_i': table["n_i"].sum(),
        'p_i': table['p_i'].sum(),
        "n'_i": table["n'_i"].sum(),
        "(n_i - n'_i)^2": table["(n_i - n'_i)^2"].sum(),
        "(n_i - n'_i)^2 / n'_i": table["(n_i - n'_i)^2 / n'_i"].sum(),
        'n_i^2': table["n_i^2"].sum(),
        "n_i^2 / n'_i": table["n_i^2 / n'_i"].sum(),
    }])
    full_table = pd.concat([table, sigma_row], ignore_index=True)

    chi_obs = float(table["(n_i - n'_i)^2 / n'_i"].sum())
    chi_check = float(table["n_i^2 / n'_i"].sum() - n)
    r = len(table)
    df = r - 3
    chi_crit = stats.chi2.ppf(0.95, df=df)

    result = {
        'name': name,
        'n': n,
        'mu': mu,
        'sigma': sigma,
        'r': r,
        'df': df,
        'chi_obs': chi_obs,
        'chi_check': chi_check,
        'chi_crit': chi_crit,
        'accept_h0': chi_obs < chi_crit,
    }
    return full_table, result


nu_chi_table, nu_chi_result = pearson_table(raw['nu'].to_numpy(), 'nu')
e_chi_table, e_chi_result = pearson_table(raw['E'].to_numpy(), 'E')

print('Таблица 1 для признака nu')
md_table(nu_chi_table)
print('Таблица 1 для признака E')
md_table(e_chi_table)


Таблица 1 для признака nu


| i   | [x_i, x_{i+1})                  |   n_i |    p_i |     n'_i |   (n_i - n'_i)^2 |   (n_i - n'_i)^2 / n'_i |   n_i^2 |   n_i^2 / n'_i |
|:----|:--------------------------------|------:|-------:|---------:|-----------------:|------------------------:|--------:|---------------:|
| 1   | (-∞, 354.12]                    |     9 | 0.0535 |   6.1576 |           8.0795 |                  1.3121 |      81 |        13.1546 |
| 2   | [354.12, 388.25)                |     4 | 0.0979 |  11.2624 |          52.743  |                  4.6831 |      16 |         1.4207 |
| 3   | [388.25, 422.38)                |    27 | 0.1753 |  20.1603 |          46.7811 |                  2.3205 |     729 |        36.1601 |
| 4   | [422.38, 456.50)                |    24 | 0.2259 |  25.9808 |           3.9236 |                  0.151  |     576 |        22.1702 |
| 5   | [456.50, 490.62)                |    24 | 0.2096 |  24.1064 |           0.0113 |                  0.0005 |     576 |        23.894  |
| 6   | [490.62, 524.75)                |    16 | 0.14   |  16.1038 |           0.0108 |                  0.0007 |     256 |        15.8969 |
| 7   | [524.75, 558.88) ∪ [558.88, +∞) |    11 | 0.0976 |  11.2287 |           0.0523 |                  0.0047 |     121 |        10.776  |
| Σ   | -                               |   115 | 1      | 115      |         111.602  |                  8.4725 |    2355 |       123.472  |

Таблица 1 для признака E


| i   | [x_i, x_{i+1})                  |   n_i |    p_i |     n'_i |   (n_i - n'_i)^2 |   (n_i - n'_i)^2 / n'_i |   n_i^2 |   n_i^2 / n'_i |
|:----|:--------------------------------|------:|-------:|---------:|-----------------:|------------------------:|--------:|---------------:|
| 1   | (-∞, 79.86] ∪ [79.86, 95.22)    |    12 | 0.0943 |  10.8467 |           1.33   |                  0.1226 |     144 |        13.2759 |
| 2   | [95.22, 110.59)                 |    16 | 0.1534 |  17.6437 |           2.7017 |                  0.1531 |     256 |        14.5094 |
| 3   | [110.59, 125.95)                |    25 | 0.2329 |  26.7806 |           3.1706 |                  0.1184 |     625 |        23.3378 |
| 4   | [125.95, 141.31)                |    30 | 0.2399 |  27.5894 |           5.811  |                  0.2106 |     900 |        32.6212 |
| 5   | [141.31, 156.68)                |    22 | 0.1678 |  19.2913 |           7.3371 |                  0.3803 |     484 |        25.089  |
| 6   | [156.68, 172.04) ∪ [172.04, +∞) |    10 | 0.1117 |  12.8483 |           8.1128 |                  0.6314 |     100 |         7.7831 |
| Σ   | -                               |   115 | 1      | 115      |          28.4632 |                  1.6165 |    2509 |       116.617  |

## 4. Проверка и вывод формулы для $\chi^2_{\text{набл}}$

Рассмотрим две записи статистики критерия Пирсона и покажем, что они совпадают:

$$
\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{(n_i - n_i')^2}{n_i'} = \sum_{i=1}^{k} \frac{n_i^2}{n_i'} - n
$$

**Вывод:**

Преобразуем первую формулу, раскрыв квадрат в числителе:

$$
\sum_{i=1}^{k} \frac{(n_i - n_i')^2}{n_i'}
=
\sum_{i=1}^{k} \frac{n_i^2 - 2n_i n_i' + (n_i')^2}{n_i'}
$$

Разделим каждое слагаемое на $n_i'$:

$$
=
\sum_{i=1}^{k} \frac{n_i^2}{n_i'} - 2\sum_{i=1}^{k} n_i + \sum_{i=1}^{k} n_i'
$$

Так как сумма наблюдаемых частот равна объёму выборки, и сумма теоретических частот также равна $n$, получаем:

$$
\sum_{i=1}^{k} n_i = n,
\qquad
\sum_{i=1}^{k} n_i' = n
$$

Следовательно,

$$
\chi^2_{\text{набл}}
=
\sum_{i=1}^{k} \frac{n_i^2}{n_i'} - 2n + n
=
\sum_{i=1}^{k} \frac{n_i^2}{n_i'} - n
\qquad \blacksquare
$$

## 5. Критическое значение и вывод

Число степеней свободы для критерия Пирсона определяется по формуле:

$$
df = k - 3
$$

Это связано с тем, что при проверке нормальности по выборке предварительно оцениваются два параметра распределения — математическое ожидание $\mu$ и среднеквадратичное отклонение $\sigma$, а также учитывается одно условие, связанное с суммой частот.

В расчётах принимается уровень значимости:

$$
\alpha = 0.05
$$

In [8]:
# Контроль вычисления χ² набл и сравнение с критическим значением
result_df = pd.DataFrame([
    {
        'Признак': nu_chi_result['name'],
        'r': nu_chi_result['r'],
        'df': nu_chi_result['df'],
        'χ²_набл': nu_chi_result['chi_obs'],
        "Контроль Σ(n_i²/n_i') - n": nu_chi_result['chi_check'],
        'χ²_крит (α=0.05)': nu_chi_result['chi_crit'],
        'Вывод': 'H0 не отвергается' if nu_chi_result['accept_h0'] else 'H0 отвергается',
    },
    {
        'Признак': e_chi_result['name'],
        'r': e_chi_result['r'],
        'df': e_chi_result['df'],
        'χ²_набл': e_chi_result['chi_obs'],
        "Контроль Σ(n_i²/n_i') - n": e_chi_result['chi_check'],
        'χ²_крит (α=0.05)': e_chi_result['chi_crit'],
        'Вывод': 'H0 не отвергается' if e_chi_result['accept_h0'] else 'H0 отвергается',
    }
])

md_table(result_df)

for res in [nu_chi_result, e_chi_result]:
    print(f"{res['name']}: χ²_набл = {res['chi_obs']:.4f}, χ²_крит = {res['chi_crit']:.4f}, df = {res['df']}")

| Признак   |   r |   df |   χ²_набл |   Контроль Σ(n_i²/n_i') - n |   χ²_крит (α=0.05) | Вывод             |
|:----------|----:|-----:|----------:|----------------------------:|-------------------:|:------------------|
| nu        |   7 |    4 |    8.4725 |                      8.4725 |             9.4877 | H0 не отвергается |
| E         |   6 |    3 |    1.6165 |                      1.6165 |             7.8147 | H0 не отвергается |

nu: χ²_набл = 8.4725, χ²_крит = 9.4877, df = 4
E: χ²_набл = 1.6165, χ²_крит = 7.8147, df = 3



По обоим признакам наблюдаемое значение критерия меньше критического, поэтому на уровне значимости $\alpha=0.05$ нет оснований отвергать гипотезу о нормальном распределении. Для признака **E** согласование с нормальной моделью выражено сильнее, так как значение $\chi^2_{набл}$ заметно меньше критического. Для признака **nu** отклонение от нормальной модели больше, но оно всё ещё не достигает критического уровня.


## Выводы

В работе были построены интервальные оценки для математического ожидания и среднеквадратичного отклонения признаков **nu** и **E** при двух уровнях доверия: $\gamma=0.95$ и $\gamma=0.99$. Во всех случаях увеличение доверительной вероятности привело к расширению интервалов, что соответствует теоретическим положениям математической статистики.

Для признака **nu** получены более широкие интервалы как для математического ожидания, так и для СКО, чем для признака **E**, поскольку объёмный вес имеет больший абсолютный разброс наблюдений. При этом центры доверительных интервалов совпадают с соответствующими выборочными средними.

Гипотеза о нормальном распределении проверялась критерием Пирсона $\chi^2$. После объединения интервалов с малыми теоретическими частотами были вычислены наблюдаемые значения статистики и выполнен контроль по формуле
$$\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n.$$
Контроль совпал с прямым вычислением, что подтверждает корректность расчётов.

На уровне значимости $\alpha=0.05$ гипотеза о нормальном распределении **не отвергается** как для **nu**, так и для **E**. Следовательно, использование нормальной модели для дальнейшего статистического анализа этих признаков является допустимым.